# Import

In [75]:
import pandas as pd

import geopandas as gpd
import seaborn as sns

import numpy as np

from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt 

from scipy import stats

# Data Wrangling

In [2]:
overlap = gpd.read_file('data\interm\mine_indig_footprint.gpkg')

I had an error in the previouse code. The mine overlay area was only multiplied with 10**-6 however 10**-9 is necessary - > multiply with 10**-3 + rename stuff. 

In [51]:
over_clean = overlap.rename(columns={'area': 'mine_area_km2', 'area_km2':'area_overlap_km2'}, inplace=False)	



Consistency check. 

* buffer 0 area mine must be larger or equal then area indig overlap 
* for buffer > larger then the radius

In [52]:
over_clean.head()

,ID,mine_area_km2,admin,adm0_a3,list_of_commodities,area_overlap_km2,buffer,geometry
0,23,0.040195,Australia,AUS,Coal,0.040131,0,"POLYGON Z ((14729392.922 -3395167.129 0.000, 1..."
1,30,0.372366,Australia,AUS,"Copper,U3O8,Gold,Zinc,Lead,Silver,Platinum,Nic...",0.065049,0,MULTIPOLYGON Z (((13644203.534 -3875200.456 0....
2,36,0.048309,Australia,AUS,"U3O8,Copper,Molybdenum,Lithium,Tungsten,Niobiu...",0.048257,0,"POLYGON Z ((13467053.269 -3679156.546 0.000, 1..."
3,37,0.242487,Australia,AUS,"U3O8,Copper,Molybdenum,Lithium,Tungsten,Niobiu...",0.242234,0,"POLYGON Z ((13465345.462 -3679101.220 0.000, 1..."
4,38,0.172674,Australia,AUS,"U3O8,Copper,Molybdenum,Lithium,Tungsten,Niobiu...",0.172483,0,"POLYGON Z ((13465248.975 -3671541.182 0.000, 1..."


In [54]:
assert all(over_clean[over_clean['buffer'] == 0]['mine_area_km2']  > over_clean[over_clean['buffer'] == 0]['area_overlap_km2'])

AssertionError: 

In [55]:
delta_0 = over_clean[over_clean['buffer'] == 0]['mine_area_km2'] - over_clean[over_clean['buffer'] == 0]['area_overlap_km2']

In [87]:
delta_0_relative = delta_0 / over_clean[over_clean['buffer'] == 0]['mine_area_km2'] *100

In [88]:
delta_sub = abs(delta_0_relative[delta_0_relative < 0])

In [90]:
# Perform a one-sample t-test
# Null hypothesis: mean(delta_0_relative) >= 1%
# Alternative hypothesis: mean(delta_0_relative) < 1%


_stat, p_value = stats.ttest_1samp(delta_sub, 1, alternative='less')

# Display the test statistic and p-value
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

# Determine if we reject the null hypothesis
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis: The mean of delta_0_relative is significantly less than 1%")
else:
    print("Fail to reject the null hypothesis: We do not have enough evidence to say the mean of delta_0_relative is less than 1%")

T-statistic: -7.816648097425087
P-value: 0.0
Reject the null hypothesis: The mean of delta_0_relative is significantly less than 1%


we can conclude that the probability that for buffer one the deviation relative to the mine area of the overlay is smaller than 1 percent is 93% this means most of the overlap calculation vary only very minor from the max of the mining area. 

# Modelling
